<div style="text-align: center; padding: 40px 0; border-bottom: 2px solid #e0e0e0; margin-bottom: 20px;">
  <h1 style="font-size: 2.5em; font-weight: 700; color: #000000; margin: 0;">
    🇧🇪 Belgium Public Transport
  </h1>
  <h2 style="font-size: 1.4em; font-weight: 300; color: #555; margin: 10px 0 0 0;">
    GTFS Dataset Analysis — 2026
  </h2>
  <p style="color: #999; margin-top: 12px; font-size: 0.9em;">
    GTFS Source: gtfs.irail.be · Provider: iRail (community) · Schedules: SNCB/NMBS · Format: GTFS only — NeTEx not obtained
  </p>
</div>


# Part 1: GTFS Exploration

## Data Sources

- **GTFS:** `gtfs-nmbs-2026-05-31.zip` from https://gtfs.irail.be/nmbs/gtfs/, maintained by iRail
  - Covers SNCB/NMBS national rail network
  - Downloaded: 31 May 2026
  - License: iRail open data — based on SNCB/NMBS official timetable data
  - **Note:** iRail is a community-driven open transport data initiative for Belgium, not a direct official SNCB publication. Data is sourced from SNCB's official timetable

- **NeTEx:** Not obtained — SNCB provides NeTEx in the Belgian EPIP profile via `belgiantrain.be`, but access requires completing a registration form and signing a licence agreement. A request was submitted and received no response. While the data formally exists, the lack of open access represents a practical barrier to its use

All third-party and standard-library imports used in this notebook are consolidated here.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

In [1]:
# Set path

data_dir = Path("data")

gtfs_zip_path = data_dir / "gtfs-nmbs-2026-05-31.zip"

print("GTFS exists:", gtfs_zip_path.exists(), gtfs_zip_path)

GTFS exists: True data/gtfs-nmbs-2026-05-31.zip


In [2]:
# Inspect GTFS ZIP contents
with zipfile.ZipFile(gtfs_zip_path, "r") as z:
    gtfs_files = pd.DataFrame([
        {
            "name":    info.filename,
            "size_mb": round(info.file_size / (1024 * 1024), 2)
        }
        for info in z.infolist()
    ])

display(gtfs_files.sort_values("size_mb", ascending=False))

,name,size_mb
7,stop_times.txt,54.87
5,stop_time_overrides.txt,34.39
4,calendar_dates.txt,25.76
0,trips.txt,4.12
3,calendar.txt,0.68
10,stops.txt,0.15
2,translations.txt,0.11
6,transfers.txt,0.07
1,routes.txt,0.06
8,agency.txt,0.00


In [3]:
# Helper function to read GTFS tables from the ZIP

def read_gtfs_from_zip(zip_path: Path, filename: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

        # If the exact file name is present, use it directly
        if filename in names:
            target = filename
        else:
            # Otherwise, search for it anywhere inside the ZIP
            matches = [n for n in names if n.lower().endswith("/" + filename.lower())]

            if not matches:
                raise FileNotFoundError(f"{filename} not found in ZIP. Example entries: {names[:20]}")
            if len(matches) > 1:
                raise FileNotFoundError(f"Multiple matches found for {filename}: {matches}")

            target = matches[0]

        with z.open(target) as f:
            return pd.read_csv(f, low_memory=False)

In [4]:
# Load core GTFS tables
stops          = read_gtfs_from_zip(gtfs_zip_path, "stops.txt")
routes         = read_gtfs_from_zip(gtfs_zip_path, "routes.txt")
trips          = read_gtfs_from_zip(gtfs_zip_path, "trips.txt")
calendar_dates = read_gtfs_from_zip(gtfs_zip_path, "calendar_dates.txt")
calendar = read_gtfs_from_zip(gtfs_zip_path, "calendar.txt")
stop_time_overrides = read_gtfs_from_zip(gtfs_zip_path, "stop_time_overrides.txt")

print("stops:",          stops.shape)
print("routes:",         routes.shape)
print("trips:",          trips.shape)
print("calendar_dates:", calendar_dates.shape)
print("calendar:" , calendar.shape)

stops: (2790, 12)
routes: (1266, 9)
trips: (52782, 9)
calendar_dates: (1500635, 3)
calendar: (18283, 10)


In [5]:
print("stops columns:")
print(list(stops.columns))
display(stops.head())

print("routes columns:")
print(list(routes.columns))
display(routes.head())

print("trips columns:")
print(list(trips.columns))
display(trips.head())

print("calendar_dates columns:")
print(list(calendar_dates.columns))
display(calendar_dates.head())

print("calendar columns:")
print(list(calendar.columns))
display(calendar.head())

print("stop_time_overrides:")
display(stop_time_overrides.head())

stops columns:
['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'zone_id', 'stop_url', 'location_type', 'parent_station', 'platform_code', 'stop_timezone']


,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,platform_code,stop_timezone
0,8011068,NaN,Frankfurt Main (DE),NaN,50.116666,8.683333,NaN,NaN,0,NaN,NaN,NaN
1,8015190,NaN,Herzogenrath,NaN,50.870900,6.094400,NaN,NaN,0,NaN,NaN,NaN
2,8015199,NaN,Aix-la-Chapelle Ouest,NaN,50.780780,6.070550,NaN,NaN,0,NaN,NaN,NaN
3,8015321,NaN,KOLN-EHRENFELD,NaN,50.951538,6.914889,NaN,NaN,0,NaN,NaN,NaN
4,8015345,NaN,Aachen Hbf (DE),NaN,50.767760,6.091390,NaN,NaN,0,NaN,NaN,NaN


routes columns:
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_desc', 'route_type', 'route_url', 'route_color', 'route_text_color']


,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color
0,1,NMBS/SNCB,IC,Binche -- Braine-le-Comte,NaN,103,NaN,NaN,NaN
1,10,NMBS/SNCB,BUS,Breda (NL) -- Noorderkempen (Brecht),NaN,700,NaN,NaN,NaN
2,100,NMBS/SNCB,IC,Bruxelles-Midi -- Namur,NaN,103,NaN,NaN,NaN
3,1000,NMBS/SNCB,BUS,Lichtervelde -- La Panne,NaN,700,NaN,NaN,NaN
4,1001,NMBS/SNCB,BUS,Ottignies -- Wavre,NaN,700,NaN,NaN,NaN


trips columns:
['route_id', 'service_id', 'trip_id', 'trip_headsign', 'trip_short_name', 'direction_id', 'block_id', 'shape_id', 'trip_type']


...

,service_id,date,exception_type
0,1,20260806,1
1,1,20260807,1
2,1,20260810,1
3,1,20260811,1
4,1,20260812,1


calendar columns:
['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,1,0,0,0,0,0,0,0,20260330,20261212
1,2,0,0,0,0,0,0,0,20260330,20261212
2,3,0,0,0,0,0,0,0,20260330,20261212
3,4,0,0,0,0,0,0,0,20260330,20261212
4,5,0,0,0,0,0,0,0,20260330,20261212


stop_time_overrides:


,trip_id,stop_sequence,service_id,stop_id
0,88____:007::8882362:8883006:6:603:20260821,1,1,8882362_1
1,88____:007::8882362:8883006:6:603:20260821,2,1,8882339_1
2,88____:007::8882362:8883006:6:603:20260821,3,1,8882206_4
3,88____:007::8882362:8883006:6:603:20260821,4,1,8882107_3
4,88____:007::8882362:8883006:6:603:20260821,5,1,8883212_2


## Comment


- The feed contains the main GTFS files needed for inspection, including `stops.txt`, `routes.txt`, `trips.txt`, `stop_times.txt`, `calendar.txt`, and `calendar_dates.txt`

- The largest files are `stop_times.txt`, `stop_time_overrides.txt`, and `calendar_dates.txt`. This shows that the feed contains detailed trip, stop-time, and service-date information.

- The main GTFS tables were loaded successfully:
  - `stops`: 2,790 rows
  - `routes`: 1,266 rows
  - `trips`: 52,782 rows
  - `calendar_dates`: 1,500,635 rows
  - `calendar`: 18,283 rows

- The stops table includes stop IDs, stop names, coordinates, `location_type`, and parent-station information

- The routes table includes route IDs, route short names, route long names, and route types. The examples show both train services such as `IC` and bus services such as `BUS`

- The trips table includes useful trip-level information such as `service_id`, `trip_id`, `trip_headsign`, and `trip_short_name`


The feed also contains `stop_time_overrides.txt`, which is not part of the official GTFS specification. It contains columns `trip_id`, `stop_sequence`, `service_id` and `stop_id`, suggesting it defines alternative stop sequences for specific trips on specific service days likely used to handle diversions or service exceptions. This file will not be used in this analysis.



## 1. Stations

The first part inspects the station information.

The goal is to check:

- how many stop rows are included
- whether stop IDs and stop names are available
- whether coordinates are available
- how `location_type` is used
- whether stops are linked to parent stations

In [6]:
# Inspect GTFS stops / stations

station_summary = pd.DataFrame({
    "metric": [
        "Total stop rows",
        "Unique stop IDs",
        "Unique stop names",
        "Rows with missing stop name",
        "Rows with missing coordinates",
        "Unique location_type values",
        "Rows with parent_station"
    ],
    "value": [
        len(stops),
        stops["stop_id"].nunique(),
        stops["stop_name"].nunique(dropna=True),
        stops["stop_name"].isna().sum(),
        stops[["stop_lat", "stop_lon"]].isna().any(axis=1).sum(),
        stops["location_type"].nunique(dropna=True) if "location_type" in stops.columns else "column missing",
        stops["parent_station"].notna().sum() if "parent_station" in stops.columns else "column missing"
    ]
})

station_summary

,metric,value
0,Total stop rows,2790
1,Unique stop IDs,2790
2,Unique stop names,675
3,Rows with missing stop name,0
4,Rows with missing coordinates,0
5,Unique location_type values,2
6,Rows with parent_station,2115


In [7]:
# Check location_type distribution

location_type_distribution = (
    stops["location_type"]
    .value_counts(dropna=False)
    .reset_index()
)

location_type_distribution.columns = ["location_type", "n_rows"]

location_type_distribution

,location_type,n_rows
0,0,2236
1,1,554


`0` = normal stop/platform

`1` = parent station

In [8]:
# Inspect stop examples

stops[
    ["stop_id", "stop_name", "stop_lat", "stop_lon", "location_type", "parent_station"]
].head(20)

,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station
0,8011068,Frankfurt Main (DE),50.116666,8.683333,0,NaN
1,8015190,Herzogenrath,50.870900,6.094400,0,NaN
2,8015199,Aix-la-Chapelle Ouest,50.780780,6.070550,0,NaN
3,8015321,KOLN-EHRENFELD,50.951538,6.914889,0,NaN
4,8015345,Aachen Hbf (DE),50.767760,6.091390,0,NaN
5,8015458,Köln Hbf (DE),50.943080,6.958970,0,NaN
6,8015485,Bonn Hbf,50.731972,7.097204,0,NaN
7,8015577,Bonn-Beuel (DE),50.738500,7.127600,0,NaN
8,8015588,Siegburg (DE),50.793585,7.203076,0,NaN
9,8015714,KOLN SUD,50.927100,6.938600,0,NaN


## Comment

The Belgian GTFS feed contains **2,790 stop rows** and they all have unique stop IDs.


- The feed contains **675 unique stop names**, so many stop names appear more than once. This is expected because one station can have several stop/platform entries.

- There are **no missing stop names** and **no missing coordinates**

- The `location_type` column has two values:
  - `0`: normal stop or platform
  - `1`: parent station

- There are **2,236 rows** with `location_type = 0` and **554 rows** with `location_type = 1`

- The feed also has **2,115 rows with a `parent_station` value**, meaning that many stop/platform entries are linked to a parent station

- This shows that the Belgian GTFS feed uses a clear station hierarchy: individual stops/platforms are grouped under parent stations

- The examples also show some international stations, such as `Frankfurt Main (DE)`, `Aachen Hbf (DE)`, `Wien HBF (AT)`, and `Linz Hbf (AT)`. This suggests that the SNCB/NMBS feed includes foreign stations served by Belgian international train services.

- Overall, the Belgian GTFS station data looks complete and well structured.


## 2. Routes

The next part inspects the route information in this feed.
The goal is to check:

- how many route rows are included
- whether route IDs are unique
- whether route names are available
- which `route_type` values appear
- whether the feed contains only rail routes or also other transport types

In [9]:
# Inspect GTFS routes

route_summary = pd.DataFrame({
    "metric": [
        "Total route rows",
        "Unique route IDs",
        "Unique route short names",
        "Unique route long names",
        "Rows with missing route_short_name",
        "Rows with missing route_long_name",
        "Unique route_type values"
    ],
    "value": [
        len(routes),
        routes["route_id"].nunique(),
        routes["route_short_name"].nunique(dropna=True),
        routes["route_long_name"].nunique(dropna=True),
        routes["route_short_name"].isna().sum(),
        routes["route_long_name"].isna().sum(),
        routes["route_type"].nunique(dropna=True)
    ]
})

route_summary

,metric,value
0,Total route rows,1266
1,Unique route IDs,1266
2,Unique route short names,36
3,Unique route long names,842
4,Rows with missing route_short_name,0
5,Rows with missing route_long_name,0
6,Unique route_type values,4


In [10]:
route_type_labels = {
    "0": "Tram",
    "1": "Metro",
    "2": "Rail",
    "3": "Bus",
    "4": "Ferry",
    "5": "Cable Tram",
    "6": "Aerial Lift",
    "7": "Funicular",
    "11": "Trolleybus",
    "12": "Monorail",
    # Extended route types
    "100": "Rail",
    "101": "High Speed Rail",
    "102": "Long Distance Rail",
    "103": "Inter Regional Rail",
    "104": "Car Transport Rail",
    "105": "Sleeper Rail",
    "106": "Regional Rail",
    "107": "Tourist Rail",
    "108": "Rail Shuttle",
    "109": "Suburban Rail",
    "400": "Urban Rail",
    "401": "Metro",
    "700": "Bus",
    "701": "Regional Bus",
    "702": "Express Bus",
    "900": "Tram",
}

route_type_distribution = (
    routes["route_type"]
    .value_counts(dropna=False)
    .reset_index()
)
route_type_distribution.columns = ["route_type", "n_routes"]
route_type_distribution["route_type"] = route_type_distribution["route_type"].astype(int).astype(str)
route_type_distribution["mode"] = route_type_distribution["route_type"].map(route_type_labels).fillna("Unknown")
route_type_distribution = route_type_distribution[["route_type", "mode", "n_routes"]]
display(route_type_distribution)

,route_type,mode,n_routes
0,103,Inter Regional Rail,638
1,100,Rail,411
2,700,Bus,216
3,101,High Speed Rail,1


- GTFS extended route types specification, maintained by Google: https://developers.google.com/transit/gtfs/reference/extended-route-types

In [11]:
# Inspect route examples

routes[
    ["route_id", "route_short_name", "route_long_name", "route_type"]
].head(20)

,route_id,route_short_name,route_long_name,route_type
0,1,IC,Binche -- Braine-le-Comte,103
1,10,BUS,Breda (NL) -- Noorderkempen (Brecht),700
2,100,IC,Bruxelles-Midi -- Namur,103
3,1000,BUS,Lichtervelde -- La Panne,700
4,1001,BUS,Ottignies -- Wavre,700
5,1002,BUS,Nivelles -- Luttre,700
6,1003,BUS,Bruxelles-Midi -- Charleroi-Central,700
7,1004,BUS,Opwijk -- Termonde,700
8,1005,BUS,Hal -- Grammont,700
9,1006,BUS,Hal -- Braine-le-Comte,700


## Comment

- The Belgian GTFS feed contains **1,266 route rows**.

- All route IDs are unique, because there are also **1,266 unique route IDs**

- There are no missing route names:
  - **0 missing `route_short_name` values**
  - **0 missing `route_long_name` values**

- The feed contains only **36 unique route short names**, but **842 unique route long names**

- This means that short names are reused a lot. For example, many routes have short names such as `IC` or `BUS`, while the long name gives the actual corridor, such as `Bruxelles-Midi -- Namur` or `Lichtervelde -- La Panne`

- The `route_type` column has four values:
  - `103`: Inter Regional Rail, **638 routes**
  - `100`: Rail, **411 routes**
  - `700`: Bus, **216 routes**
  - `101`: High Speed Rail, **1 route**

- Most routes are therefore rail-related, but the feed also includes bus routes

Overall, the route table is complete and usable. However, it is important to note that Belgium uses more detailed GTFS route type codes, and the route short names are broad labels rather than unique route identifiers.



## 3. Calendar

The next part inspects the calendar information.

The goal is to check:

- how many service rows are included in `calendar.txt`
- how many service-date exception rows are included in `calendar_dates.txt`
- what date range is covered
- whether service IDs used in `trips.txt` also appear in the calendar tables
- which `exception_type` values are used

In [12]:
# Prepare GTFS calendar dates

calendar["start_date"] = pd.to_datetime(
    calendar["start_date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

calendar["end_date"] = pd.to_datetime(
    calendar["end_date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

calendar_dates["date"] = pd.to_datetime(
    calendar_dates["date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

trip_service_ids = set(trips["service_id"].dropna().astype(str))
calendar_service_ids = set(calendar["service_id"].dropna().astype(str))
calendar_dates_service_ids = set(calendar_dates["service_id"].dropna().astype(str))

all_calendar_service_ids = calendar_service_ids | calendar_dates_service_ids

calendar_summary = pd.DataFrame({
    "metric": [
        "calendar rows",
        "calendar_dates rows",
        "Unique service IDs in trips",
        "Unique service IDs in calendar",
        "Unique service IDs in calendar_dates",
        "Service IDs in trips missing from calendar/calendar_dates",
        "Service IDs in calendar/calendar_dates not used in trips",
        "Invalid start_date rows",
        "Invalid end_date rows",
        "Invalid calendar_dates date rows",
        "Calendar start date",
        "Calendar end date",
        "calendar_dates start date",
        "calendar_dates end date",
        "Unique exception_type values"
    ],
    "value": [
        len(calendar),
        len(calendar_dates),
        len(trip_service_ids),
        len(calendar_service_ids),
        len(calendar_dates_service_ids),
        len(trip_service_ids - all_calendar_service_ids),
        len(all_calendar_service_ids - trip_service_ids),
        calendar["start_date"].isna().sum(),
        calendar["end_date"].isna().sum(),
        calendar_dates["date"].isna().sum(),
        calendar["start_date"].min(),
        calendar["end_date"].max(),
        calendar_dates["date"].min(),
        calendar_dates["date"].max(),
        calendar_dates["exception_type"].nunique(dropna=True)
    ]
})

calendar_summary

,metric,value
0,calendar rows,18283
1,calendar_dates rows,1500635
2,Unique service IDs in trips,3454
3,Unique service IDs in calendar,18283
4,Unique service IDs in calendar_dates,18283
5,Service IDs in trips missing from calendar/cal...,0
6,Service IDs in calendar/calendar_dates not use...,14829
7,Invalid start_date rows,0
8,Invalid end_date rows,0
9,Invalid calendar_dates date rows,0


In [13]:
# Check exception_type distribution

exception_type_distribution = (
    calendar_dates["exception_type"]
    .value_counts(dropna=False)
    .reset_index()
)

exception_type_distribution.columns = ["exception_type", "n_rows"]

exception_type_distribution

,exception_type,n_rows
0,1,1500635


- `1` = service is added / operates on that date

- `2` = service is removed / does not operate on that date

## Comment

- The Belgian GTFS feed contains both `calendar.txt` and `calendar_dates.txt`

- `calendar.txt` contains **18,283 rows**

- `calendar_dates.txt` contains **1,500,635 rows**, so the feed has a very detailed service-date table

- There are **3,454 unique service IDs** used in `trips.txt`

- All service IDs used by trips are also present in the calendar information. This means there are **no trip service IDs missing from the calendar tables**

- There are **14,829 service IDs** that appear in the calendar information but are not used in `trips.txt`. This suggests that the calendar tables contain more service patterns than the trips table actually uses

- There are no invalid dates in `calendar.txt` or `calendar_dates.txt`

- Both calendar tables cover the same period: **2026-03-30 to 2026-12-12**

- Only one `exception_type` value appears in `calendar_dates.txt`:
  - `1`: service is added / operates on that date

- There are no `exception_type = 2` rows, so removed services are not listed in this file

Overall, the Belgian GTFS calendar data is complete for the services used in trips, but the calendar tables contain many additional service IDs that are not used by trips.



| Part | What was checked | Main finding | Interpretation |
|---|---|---|---|
| Data availability | GTFS and NeTEx access | GTFS was available from the iRail `nmbs` folder. For NeTEx, access required registration/licence approval. I contacted the provider, submitted the signed form, but did not receive an answer. | Belgium can be inspected on the GTFS side, but a full GTFS–NeTEx comparison is not possible. |
| Additional files | Non-core GTFS files | The feed contains `stop_time_overrides.txt`, which is not part of the core GTFS files used in this inspection. | The file is noted but not analysed further, because the inspection focuses on the standard GTFS structure. |
| Stations | Stop IDs, names, coordinates, `location_type`, and `parent_station` | 2,790 stop rows, no missing names, no missing coordinates, 554 parent stations, and 2,115 rows linked to a parent station. | The station data is complete and well structured on the GTFS side. |
| Routes | Route IDs, names, and `route_type` | 1,266 routes, no missing route names, and four route type values: rail, inter-regional rail, high-speed rail, and bus. | The route data is complete, but the feed uses detailed route type codes and includes both rail and bus routes. |
| Calendar | `calendar.txt`, `calendar_dates.txt`, service IDs, date range, and `exception_type` | All trip service IDs are covered by the calendar tables. The date range is 2026-03-30 to 2026-12-12. | Calendar data is complete for the services used in trips, but the calendar tables contain many additional service IDs not used by trips. |
| Overall result | GTFS-only inspection | The GTFS feed is usable and detailed, but the NeTEx dataset could not be accessed. | Belgium is documented as a GTFS-only or limited-comparison case unless NeTEx access becomes available. |